# 🍜 Miso Agent Team Demo

## 4-Agent Software Development Team

This demo showcases **miso**'s `Team` + `Agent` multi-agent collaboration system.

### Team Composition
| Agent | Role | Emoji |
|-------|------|-------|
| **PM** (owner) | Product Manager — decomposes requirements, assigns tasks, delivers final summary | 📋 |
| **FrontendDev** | Frontend Developer — implements UI components | 🎨 |
| **BackendDev** | Backend Developer — designs APIs & server logic | ⚙️ |
| **TestEngineer** | Test Engineer — writes test plans & validates deliverables | 🧪 |

### Architecture
```
User Task ──▶ [shared channel] ──▶ PM decomposes ──▶ FE + BE work ──▶ QA validates ──▶ PM finalizes
```

Two modes available:
- **🤖 Simulated Mode** (default) — no API key needed, uses scripted high-quality responses
- **🔥 LLM Mode** (optional) — set your API key to use real LLM calls

---
## Cell 1 — 📦 Imports & Setup

In [1]:
import sys, os, json, time, copy, html
from datetime import datetime
from IPython.display import display, HTML, clear_output

# Make sure miso is importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

from miso import Agent, Team

print("✅ miso imported successfully")
print(f"   Agent: {Agent}")
print(f"   Team:  {Team}")

✅ miso imported successfully
   Agent: <class 'miso.agent.Agent'>
   Team:  <class 'miso.team.Team'>


---
## Cell 2 — 🏗️ Define the 4 Agents

In [2]:
# ── Agent Definitions ──────────────────────────────────────────────

pm = Agent(
    name="PM",
    instructions="""
You are the Product Manager. Your responsibilities:
1. Receive user requirements and decompose them into a clear PRD (Product Requirements Document).
2. Assign frontend tasks to @FrontendDev and backend tasks to @BackendDev.
3. After both developers finish, ask @TestEngineer to validate.
4. After testing is complete, produce the final summary for the user.
Be concise, structured, and professional.
""",
)

frontend_dev = Agent(
    name="FrontendDev",
    instructions="""
You are the Frontend Developer. Your responsibilities:
1. Read the PRD from PM and implement the UI components.
2. Use React + TypeScript best practices.
3. Publish your completed code/design back to the shared channel.
Be concise and output working code.
""",
)

backend_dev = Agent(
    name="BackendDev",
    instructions="""
You are the Backend Developer. Your responsibilities:
1. Read the PRD from PM and design the API endpoints + data models.
2. Use Python FastAPI best practices.
3. Publish your completed API design and code back to the shared channel.
Be concise and output working code.
""",
)

test_engineer = Agent(
    name="TestEngineer",
    instructions="""
You are the Test Engineer. Your responsibilities:
1. Wait for both FrontendDev and BackendDev to complete their work.
2. Review their outputs and create a comprehensive test plan.
3. Report test results (pass/fail) back to the shared channel.
Be thorough but concise.
""",
)

agents = [pm, frontend_dev, backend_dev, test_engineer]
print("✅ 4 Agents created:")
for a in agents:
    owner_tag = " 👑 (owner)" if a.name == "PM" else ""
    print(f"   • {a.name}{owner_tag}")

✅ 4 Agents created:
   • PM 👑 (owner)
   • FrontendDev
   • BackendDev
   • TestEngineer


---
## Cell 3 — 🔧 Simulated Responses (No API Key Needed)

We monkeypatch each agent's `step()` method with carefully crafted responses that simulate a realistic collaboration flow.

In [3]:
# ── Helper to build step results (same shape as Agent.step() output) ──

def _step_result(*, publish=None, handoff=None, final="", idle=False, artifacts=None):
    return {
        "agent": "",
        "publish": publish or [],
        "handoff": handoff,
        "final": final,
        "idle": idle,
        "artifacts": artifacts or [],
        "conversation": [],
        "bundle": {},
        "raw_output": {},
    }


# ── PM Simulated Behavior ──────────────────────────────────────────
pm_call_count = 0

def pm_step(**kwargs):
    global pm_call_count
    pm_call_count += 1
    
    if pm_call_count == 1:
        # First call: receive user task, decompose into PRD
        return _step_result(publish=[{
            "channel": "shared",
            "content": """## 📋 PRD: Todo List Web Application

### Overview
Build a full-stack Todo List web app with CRUD operations.

### Frontend Requirements (@FrontendDev)
- React + TypeScript SPA
- Components: TodoList, TodoItem, AddTodoForm, FilterBar
- State management with React hooks
- Responsive design with Tailwind CSS
- Features: add, toggle complete, delete, filter (all/active/completed)

### Backend Requirements (@BackendDev)
- Python FastAPI REST API
- Endpoints: GET /todos, POST /todos, PUT /todos/{id}, DELETE /todos/{id}
- SQLite database with SQLAlchemy ORM
- Pydantic models for request/response validation
- CORS middleware for frontend integration

### Timeline
FrontendDev and BackendDev work in parallel → TestEngineer validates → PM delivers final report.

@FrontendDev @BackendDev Please begin implementation.""",
            "mentions": ["FrontendDev", "BackendDev"],
        }])
    
    elif pm_call_count == 2:
        # Second call: after devs finish, ask QA
        return _step_result(publish=[{
            "channel": "shared",
            "content": """Both frontend and backend implementations look great! 🎉

@TestEngineer Please review the deliverables from FrontendDev and BackendDev, and run your test plan.""",
            "mentions": ["TestEngineer"],
        }])
    
    else:
        # Third call: after QA, finalize
        return _step_result(final="""## ✅ Project Complete: Todo List Web Application

### Summary
All components have been implemented and tested successfully.

### Deliverables
| Component | Status | Owner |
|-----------|--------|-------|
| Frontend (React + TS) | ✅ Complete | FrontendDev |
| Backend (FastAPI) | ✅ Complete | BackendDev |
| Test Suite | ✅ All Passed | TestEngineer |

### Architecture
- **Frontend**: React 18 + TypeScript + Tailwind CSS, 4 components
- **Backend**: FastAPI + SQLAlchemy + SQLite, 4 REST endpoints
- **Tests**: 12 test cases (6 API + 4 component + 2 integration), all passing

### Next Steps
1. Deploy frontend to Vercel
2. Deploy backend to Railway/Fly.io
3. Set up CI/CD pipeline

The project is ready for deployment! 🚀""")


# ── FrontendDev Simulated Behavior ─────────────────────────────────
def frontend_step(**kwargs):
    return _step_result(publish=[{
        "channel": "shared",
        "content": """## 🎨 Frontend Implementation Complete

### Component Structure
```
src/
├── App.tsx
├── components/
│   ├── TodoList.tsx
│   ├── TodoItem.tsx
│   ├── AddTodoForm.tsx
│   └── FilterBar.tsx
├── hooks/
│   └── useTodos.ts
├── types/
│   └── todo.ts
└── api/
    └── todoApi.ts
```

### Key Code — TodoList.tsx
```tsx
import React from 'react';
import { useTodos } from '../hooks/useTodos';
import { TodoItem } from './TodoItem';
import { AddTodoForm } from './AddTodoForm';
import { FilterBar } from './FilterBar';

export const TodoList: React.FC = () => {
  const { todos, filter, addTodo, toggleTodo, deleteTodo, setFilter } = useTodos();
  const filtered = todos.filter(t =>
    filter === 'all' ? true : filter === 'active' ? !t.completed : t.completed
  );

  return (
    <div className=\"max-w-md mx-auto p-6\">
      <h1 className=\"text-2xl font-bold mb-4\">📝 Todo List</h1>
      <AddTodoForm onAdd={addTodo} />
      <FilterBar current={filter} onChange={setFilter} />
      <ul className=\"space-y-2\">
        {filtered.map(todo => (
          <TodoItem key={todo.id} todo={todo} onToggle={toggleTodo} onDelete={deleteTodo} />
        ))}
      </ul>
      <p className=\"text-sm text-gray-500 mt-4\">{todos.filter(t => !t.completed).length} items left</p>
    </div>
  );
};
```

### Features Implemented
- ✅ Add new todos with Enter key
- ✅ Toggle completion with checkbox
- ✅ Delete with trash icon
- ✅ Filter: All / Active / Completed
- ✅ Responsive Tailwind CSS design
- ✅ API integration via `todoApi.ts`

Frontend is ready for integration testing! @PM""",
        "mentions": ["PM"],
    }])


# ── BackendDev Simulated Behavior ──────────────────────────────────
def backend_step(**kwargs):
    return _step_result(publish=[{
        "channel": "shared",
        "content": """## ⚙️ Backend Implementation Complete

### Project Structure
```
backend/
├── main.py
├── models.py
├── schemas.py
├── database.py
└── requirements.txt
```

### Key Code — main.py
```python
from fastapi import FastAPI, HTTPException, Depends
from fastapi.middleware.cors import CORSMiddleware
from sqlalchemy.orm import Session
from database import get_db, engine
import models, schemas

models.Base.metadata.create_all(bind=engine)
app = FastAPI(title=\"Todo API\")

app.add_middleware(CORSMiddleware, allow_origins=[\"*\"], allow_methods=[\"*\"], allow_headers=[\"*\"])

@app.get(\"/todos\", response_model=list[schemas.TodoResponse])
def list_todos(db: Session = Depends(get_db)):
    return db.query(models.Todo).all()

@app.post(\"/todos\", response_model=schemas.TodoResponse, status_code=201)
def create_todo(todo: schemas.TodoCreate, db: Session = Depends(get_db)):
    db_todo = models.Todo(title=todo.title)
    db.add(db_todo); db.commit(); db.refresh(db_todo)
    return db_todo

@app.put(\"/todos/{todo_id}\", response_model=schemas.TodoResponse)
def update_todo(todo_id: int, todo: schemas.TodoUpdate, db: Session = Depends(get_db)):
    db_todo = db.query(models.Todo).get(todo_id)
    if not db_todo: raise HTTPException(404, \"Todo not found\")
    if todo.title is not None: db_todo.title = todo.title
    if todo.completed is not None: db_todo.completed = todo.completed
    db.commit(); db.refresh(db_todo)
    return db_todo

@app.delete(\"/todos/{todo_id}\", status_code=204)
def delete_todo(todo_id: int, db: Session = Depends(get_db)):
    db_todo = db.query(models.Todo).get(todo_id)
    if not db_todo: raise HTTPException(404, \"Todo not found\")
    db.delete(db_todo); db.commit()
```

### API Endpoints
| Method | Endpoint | Description |
|--------|----------|-------------|
| GET | /todos | List all todos |
| POST | /todos | Create a new todo |
| PUT | /todos/{id} | Update a todo |
| DELETE | /todos/{id} | Delete a todo |

### Tech Stack
- FastAPI 0.104+
- SQLAlchemy 2.0 + SQLite
- Pydantic v2 for validation
- CORS enabled for frontend integration

Backend is ready for integration testing! @PM""",
        "mentions": ["PM"],
    }])


# ── TestEngineer Simulated Behavior ────────────────────────────────
def test_step(**kwargs):
    return _step_result(publish=[{
        "channel": "shared",
        "content": """## 🧪 Test Report

### Test Plan Execution

#### API Tests (6/6 passed ✅)
| # | Test Case | Status |
|---|-----------|--------|
| 1 | GET /todos returns empty list initially | ✅ Pass |
| 2 | POST /todos creates a new todo | ✅ Pass |
| 3 | GET /todos returns created todo | ✅ Pass |
| 4 | PUT /todos/{id} toggles completion | ✅ Pass |
| 5 | DELETE /todos/{id} removes todo | ✅ Pass |
| 6 | PUT /todos/999 returns 404 | ✅ Pass |

#### Component Tests (4/4 passed ✅)
| # | Test Case | Status |
|---|-----------|--------|
| 1 | TodoList renders empty state | ✅ Pass |
| 2 | AddTodoForm creates new item on Enter | ✅ Pass |
| 3 | TodoItem toggle calls onToggle | ✅ Pass |
| 4 | FilterBar switches between filters | ✅ Pass |

#### Integration Tests (2/2 passed ✅)
| # | Test Case | Status |
|---|-----------|--------|
| 1 | Frontend ↔ Backend CRUD flow | ✅ Pass |
| 2 | Concurrent operations consistency | ✅ Pass |

### Summary
**12/12 tests passed.** All components are working correctly.
No critical bugs found. Ready for deployment. @PM""",
        "mentions": ["PM"],
    }])


# ── Patch agents with simulated step() ─────────────────────────────
pm.step = pm_step
frontend_dev.step = frontend_step
backend_dev.step = backend_step
test_engineer.step = test_step

print("✅ Simulated responses patched onto all 4 agents")
print("   (No API key needed — switch to LLM mode in the last cell)")

✅ Simulated responses patched onto all 4 agents
   (No API key needed — switch to LLM mode in the last cell)


---
## Cell 4 — 🎨 Visualization Engine

Rich HTML/CSS rendering for:
1. **Channel Timeline** — Slack-style message feed
2. **Agent Process Dashboard** — status & action log per agent
3. **Task Board** — kanban-style progress view

In [4]:
# ── Visualization Configuration ───────────────────────────────────

AGENT_CONFIG = {
    "user":         {"emoji": "👤", "color": "#6B7280", "label": "User"},
    "PM":           {"emoji": "📋", "color": "#3B82F6", "label": "PM"},
    "FrontendDev":  {"emoji": "🎨", "color": "#8B5CF6", "label": "FrontendDev"},
    "BackendDev":   {"emoji": "⚙️", "color": "#F59E0B", "label": "BackendDev"},
    "TestEngineer": {"emoji": "🧪", "color": "#10B981", "label": "TestEngineer"},
}

def _get_agent_cfg(name):
    return AGENT_CONFIG.get(name, {"emoji": "🤖", "color": "#6B7280", "label": name})


# ── CSS Styles ─────────────────────────────────────────────────────

STYLES = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

.miso-demo * { font-family: 'Inter', -apple-system, sans-serif; box-sizing: border-box; }
.miso-demo { max-width: 1100px; margin: 0 auto; }

/* ── Section Headers ── */
.miso-section-header {
    font-size: 18px; font-weight: 700; margin: 28px 0 14px 0;
    padding: 10px 16px; border-radius: 10px;
    background: linear-gradient(135deg, #1e293b, #334155);
    color: #f1f5f9; letter-spacing: 0.3px;
}

/* ── Channel Timeline ── */
.miso-timeline { display: flex; flex-direction: column; gap: 0; }
.miso-msg {
    display: flex; gap: 12px; padding: 14px 16px;
    border-bottom: 1px solid #e2e8f0; transition: background 0.15s;
}
.miso-msg:hover { background: #f8fafc; }
.miso-avatar {
    width: 40px; height: 40px; border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
    font-size: 20px; flex-shrink: 0;
}
.miso-msg-body { flex: 1; min-width: 0; }
.miso-msg-header { display: flex; align-items: baseline; gap: 8px; margin-bottom: 4px; }
.miso-msg-name { font-weight: 700; font-size: 14px; }
.miso-msg-meta { font-size: 11px; color: #94a3b8; }
.miso-msg-badge {
    font-size: 10px; padding: 1px 7px; border-radius: 4px;
    font-weight: 600; color: white;
}
.miso-msg-content {
    font-size: 13px; line-height: 1.6; color: #1e293b;
    white-space: pre-wrap; word-break: break-word;
}
.miso-msg-content code {
    background: #f1f5f9; padding: 1px 5px; border-radius: 4px;
    font-size: 12px; font-family: 'SF Mono', 'Fira Code', monospace;
}

/* ── Agent Dashboard ── */
.miso-dashboard { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; }
.miso-agent-card {
    border-radius: 12px; padding: 16px; border: 1px solid #e2e8f0;
    background: white; box-shadow: 0 1px 3px rgba(0,0,0,0.06);
}
.miso-agent-header {
    display: flex; align-items: center; gap: 8px;
    margin-bottom: 12px; padding-bottom: 10px; border-bottom: 2px solid #f1f5f9;
}
.miso-agent-emoji { font-size: 24px; }
.miso-agent-name { font-weight: 700; font-size: 14px; }
.miso-agent-role { font-size: 11px; color: #94a3b8; }
.miso-process-list { list-style: none; padding: 0; margin: 0; }
.miso-process-item {
    font-size: 12px; padding: 6px 0; color: #475569;
    border-bottom: 1px solid #f8fafc;
    display: flex; align-items: flex-start; gap: 6px;
}
.miso-process-icon { flex-shrink: 0; }
.miso-status-badge {
    display: inline-block; font-size: 11px; font-weight: 600;
    padding: 2px 10px; border-radius: 20px; margin-top: 8px;
}

/* ── Task Board ── */
.miso-board { display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; }
.miso-board-col {
    border-radius: 12px; padding: 14px; min-height: 120px;
    background: #f8fafc; border: 1px solid #e2e8f0;
}
.miso-board-title {
    font-size: 12px; font-weight: 700; text-transform: uppercase;
    letter-spacing: 0.8px; margin-bottom: 10px; padding-bottom: 8px;
    border-bottom: 2px solid #e2e8f0;
}
.miso-board-card {
    background: white; border-radius: 8px; padding: 10px 12px;
    margin-bottom: 8px; border-left: 3px solid #94a3b8;
    font-size: 12px; box-shadow: 0 1px 2px rgba(0,0,0,0.04);
}
.miso-board-card-title { font-weight: 600; margin-bottom: 3px; }
.miso-board-card-agent { font-size: 11px; color: #94a3b8; }

/* ── Summary Stats ── */
.miso-stats { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 16px; }
.miso-stat-card {
    text-align: center; padding: 16px; border-radius: 12px;
    background: white; border: 1px solid #e2e8f0;
}
.miso-stat-value { font-size: 28px; font-weight: 700; }
.miso-stat-label { font-size: 11px; color: #94a3b8; margin-top: 4px; }
</style>
"""


# ── Render Functions ────────────────────────────────────────────────

def render_stats(result):
    """Top-level summary stats."""
    events = result["events"]
    n_steps = result["steps"]
    n_messages = len([e for e in events if e["type"] == "message_published"])
    n_agents = len(set(e["agent"] for e in events if e["type"] == "scheduled"))
    stop = result["stop_reason"]
    
    return f"""
    <div class="miso-stats">
        <div class="miso-stat-card">
            <div class="miso-stat-value" style="color:#3B82F6">{n_steps}</div>
            <div class="miso-stat-label">Total Steps</div>
        </div>
        <div class="miso-stat-card">
            <div class="miso-stat-value" style="color:#8B5CF6">{n_messages}</div>
            <div class="miso-stat-label">Messages</div>
        </div>
        <div class="miso-stat-card">
            <div class="miso-stat-value" style="color:#F59E0B">{n_agents}</div>
            <div class="miso-stat-label">Active Agents</div>
        </div>
        <div class="miso-stat-card">
            <div class="miso-stat-value" style="color:#10B981">{'✅' if stop == 'owner_finalized' else '⚠️'}</div>
            <div class="miso-stat-label">{stop}</div>
        </div>
    </div>
    """


def render_channel_timeline(result):
    """Slack-style message timeline from transcript."""
    transcript = result["transcript"]
    rows = []
    
    for env in transcript:
        sender = env.get("sender", "unknown")
        cfg = _get_agent_cfg(sender)
        content = html.escape(env.get("content", ""))
        # Simple markdown-ish rendering
        content = content.replace("\n", "<br>")
        channel = env.get("channel", "")
        step = env.get("step", "?")
        kind = env.get("kind", "message")
        mentions = env.get("mentions", [])
        
        # Highlight @mentions
        for m in mentions:
            m_cfg = _get_agent_cfg(m)
            content = content.replace(
                f"@{m}",
                f'<span style="background:{m_cfg["color"]}22;color:{m_cfg["color"]};padding:1px 5px;border-radius:4px;font-weight:600;">@{m}</span>'
            )
        
        kind_badge = ""
        if kind == "task":
            kind_badge = f'<span class="miso-msg-badge" style="background:#EF4444;">TASK</span>'
        elif kind == "handoff":
            kind_badge = f'<span class="miso-msg-badge" style="background:#F59E0B;">HANDOFF</span>'
        
        rows.append(f"""
        <div class="miso-msg">
            <div class="miso-avatar" style="background:{cfg['color']}15;">{cfg['emoji']}</div>
            <div class="miso-msg-body">
                <div class="miso-msg-header">
                    <span class="miso-msg-name" style="color:{cfg['color']};">{cfg['label']}</span>
                    {kind_badge}
                    <span class="miso-msg-meta">step {step} · #{channel}</span>
                </div>
                <div class="miso-msg-content">{content}</div>
            </div>
        </div>
        """)
    
    return f"""
    <div class="miso-section-header">📺 Channel Timeline — #shared</div>
    <div class="miso-timeline" style="border:1px solid #e2e8f0;border-radius:12px;overflow:hidden;background:white;">
        {''.join(rows)}
    </div>
    """


def render_agent_dashboard(result):
    """Per-agent process list & status."""
    events = result["events"]
    
    # Build per-agent process logs
    agent_logs = {name: [] for name in ["PM", "FrontendDev", "BackendDev", "TestEngineer"]}
    agent_status = {name: "idle" for name in agent_logs}
    
    for ev in events:
        agent = ev.get("agent", "")
        if agent not in agent_logs:
            continue
        
        etype = ev["type"]
        step = ev.get("step", "?")
        
        if etype == "scheduled":
            agent_logs[agent].append(("📥", f"Step {step}: Scheduled (inbox: {ev.get('inbox_size', 0)})"))
            agent_status[agent] = "working"
        elif etype == "message_published":
            ch = ev.get("channel", "")
            preview = ev.get("content", "")[:60].replace("\n", " ")
            agent_logs[agent].append(("💬", f"Step {step}: Published to #{ch}"))
            agent_status[agent] = "done"
        elif etype == "handoff":
            agent_logs[agent].append(("🤝", f"Step {step}: Handoff → {ev.get('target', '?')}"))
        elif etype == "finalized":
            agent_logs[agent].append(("🏁", f"Step {step}: Finalized!"))
            agent_status[agent] = "finalized"
        elif etype == "idle":
            agent_logs[agent].append(("💤", f"Step {step}: Idle"))
    
    status_styles = {
        "idle": ("#94a3b8", "#f1f5f9", "⚪ IDLE"),
        "working": ("#F59E0B", "#FEF3C7", "🟡 WORKING"),
        "done": ("#10B981", "#D1FAE5", "🟢 DONE"),
        "finalized": ("#3B82F6", "#DBEAFE", "🏁 FINALIZED"),
    }
    
    role_map = {
        "PM": "Product Manager",
        "FrontendDev": "Frontend Developer",
        "BackendDev": "Backend Developer",
        "TestEngineer": "Test Engineer",
    }
    
    cards = []
    for name in ["PM", "FrontendDev", "BackendDev", "TestEngineer"]:
        cfg = _get_agent_cfg(name)
        status = agent_status[name]
        s_color, s_bg, s_label = status_styles[status]
        
        log_items = ""
        for icon, text in agent_logs[name]:
            log_items += f'<li class="miso-process-item"><span class="miso-process-icon">{icon}</span> {html.escape(text)}</li>'
        
        if not log_items:
            log_items = '<li class="miso-process-item" style="color:#cbd5e1;">No activity yet</li>'
        
        cards.append(f"""
        <div class="miso-agent-card" style="border-top: 3px solid {cfg['color']};">
            <div class="miso-agent-header">
                <span class="miso-agent-emoji">{cfg['emoji']}</span>
                <div>
                    <div class="miso-agent-name" style="color:{cfg['color']};">{name}</div>
                    <div class="miso-agent-role">{role_map.get(name, '')}</div>
                </div>
            </div>
            <ul class="miso-process-list">{log_items}</ul>
            <div class="miso-status-badge" style="color:{s_color};background:{s_bg};">{s_label}</div>
        </div>
        """)
    
    return f"""
    <div class="miso-section-header">📊 Agent Process Dashboard</div>
    <div class="miso-dashboard">{''.join(cards)}</div>
    """


def render_task_board(result):
    """Kanban-style task board."""
    events = result["events"]
    
    # Determine task states from events
    tasks = {
        "PRD & Task Decomposition": {"agent": "PM", "color": "#3B82F6", "status": "todo"},
        "Frontend Implementation": {"agent": "FrontendDev", "color": "#8B5CF6", "status": "todo"},
        "Backend Implementation": {"agent": "BackendDev", "color": "#F59E0B", "status": "todo"},
        "Testing & QA": {"agent": "TestEngineer", "color": "#10B981", "status": "todo"},
        "Final Delivery": {"agent": "PM", "color": "#3B82F6", "status": "todo"},
    }
    
    scheduled_agents = set()
    published_agents = set()
    finalized = False
    
    for ev in events:
        if ev["type"] == "scheduled":
            scheduled_agents.add(ev["agent"])
        if ev["type"] == "message_published" and ev["agent"] != "user":
            published_agents.add(ev["agent"])
        if ev["type"] == "finalized":
            finalized = True
    
    # Update statuses
    if "PM" in published_agents:
        tasks["PRD & Task Decomposition"]["status"] = "done"
    if "FrontendDev" in scheduled_agents:
        tasks["Frontend Implementation"]["status"] = "progress"
    if "FrontendDev" in published_agents:
        tasks["Frontend Implementation"]["status"] = "done"
    if "BackendDev" in scheduled_agents:
        tasks["Backend Implementation"]["status"] = "progress"
    if "BackendDev" in published_agents:
        tasks["Backend Implementation"]["status"] = "done"
    if "TestEngineer" in scheduled_agents:
        tasks["Testing & QA"]["status"] = "progress"
    if "TestEngineer" in published_agents:
        tasks["Testing & QA"]["status"] = "done"
    if finalized:
        tasks["Final Delivery"]["status"] = "done"
    elif "TestEngineer" in published_agents:
        tasks["Final Delivery"]["status"] = "progress"
    
    columns = {"todo": [], "progress": [], "done": []}
    for task_name, info in tasks.items():
        cfg = _get_agent_cfg(info["agent"])
        card = f"""
        <div class="miso-board-card" style="border-left-color:{info['color']};">
            <div class="miso-board-card-title">{task_name}</div>
            <div class="miso-board-card-agent">{cfg['emoji']} {info['agent']}</div>
        </div>
        """
        columns[info["status"]].append(card)
    
    col_configs = [
        ("📋 TODO", "todo", "#94a3b8"),
        ("🔨 IN PROGRESS", "progress", "#F59E0B"),
        ("✅ DONE", "done", "#10B981"),
    ]
    
    cols_html = []
    for title, key, color in col_configs:
        items = ''.join(columns[key]) or '<div style="font-size:12px;color:#cbd5e1;padding:8px;">No tasks</div>'
        cols_html.append(f"""
        <div class="miso-board-col">
            <div class="miso-board-title" style="color:{color};">{title} ({len(columns[key])})</div>
            {items}
        </div>
        """)
    
    return f"""
    <div class="miso-section-header">🏗️ Task Board</div>
    <div class="miso-board">{''.join(cols_html)}</div>
    """


def render_final_output(result):
    """The final answer from the owner agent."""
    final = html.escape(result.get("final", "")).replace("\n", "<br>")
    if not final:
        return ""
    return f"""
    <div class="miso-section-header">🎯 Final Output — from {result.get('final_agent', 'owner')}</div>
    <div style="background:white;border:2px solid #3B82F6;border-radius:12px;padding:20px;font-size:13px;line-height:1.7;color:#1e293b;">
        {final}
    </div>
    """


def render_all(result):
    """Render the complete visualization."""
    full_html = STYLES + '<div class="miso-demo">'
    full_html += render_stats(result)
    full_html += render_agent_dashboard(result)
    full_html += render_task_board(result)
    full_html += render_channel_timeline(result)
    full_html += render_final_output(result)
    full_html += '</div>'
    display(HTML(full_html))


print("✅ Visualization engine loaded")
print("   • render_stats()           — summary stat cards")
print("   • render_channel_timeline() — Slack-style message feed")
print("   • render_agent_dashboard()  — per-agent process list")
print("   • render_task_board()       — kanban board")
print("   • render_all()              — everything combined")

✅ Visualization engine loaded
   • render_stats()           — summary stat cards
   • render_channel_timeline() — Slack-style message feed
   • render_agent_dashboard()  — per-agent process list
   • render_task_board()       — kanban board
   • render_all()              — everything combined


---
## Cell 5 — 🚀 Run the Team!

Create the Team, submit a task, and watch the agents collaborate.

In [5]:
# ── Create the Team ────────────────────────────────────────────────

team = Team(
    agents=[pm, frontend_dev, backend_dev, test_engineer],
    owner="PM",
    channels={
        "shared": ["PM", "FrontendDev", "BackendDev", "TestEngineer"],
    },
    max_steps=24,
)

print("✅ Team created")
print(f"   Owner: {team.owner}")
print(f"   Channels: {list(team.channels.keys())}")
print(f"   Agents: {team._agent_order}")
print(f"   Max steps: {team.max_steps}")
print()

# ── Run! ────────────────────────────────────────────────────────────

USER_TASK = "Build a Todo List web application with full CRUD functionality."

# Live event callback — prints events as they happen
def live_callback(event):
    etype = event.get("type", "")
    agent = event.get("agent", "")
    step = event.get("step", "?")
    cfg = _get_agent_cfg(agent)
    
    if etype == "scheduled":
        print(f"  ⏱️  Step {step}: {cfg['emoji']} {agent} scheduled (inbox: {event.get('inbox_size', 0)})")
    elif etype == "message_published":
        preview = event.get("content", "")[:80].replace("\n", " ")
        print(f"  💬  Step {step}: {cfg['emoji']} {agent} → #{event.get('channel', '')} | {preview}...")
    elif etype == "finalized":
        print(f"  🏁  Step {step}: {cfg['emoji']} {agent} FINALIZED the project!")
    elif etype == "handoff":
        print(f"  🤝  Step {step}: {cfg['emoji']} {agent} → handoff to {event.get('target', '?')}")

print(f'📝 User Task: "{USER_TASK}"')
print("=" * 70)

result = team.run(USER_TASK, callback=live_callback)

print("=" * 70)
print(f"\n✅ Team finished!")
print(f"   Stop reason: {result['stop_reason']}")
print(f"   Total steps: {result['steps']}")
print(f"   Transcript messages: {len(result['transcript'])}")
print(f"   Events: {len(result['events'])}")

✅ Team created
   Owner: PM
   Channels: ['shared']
   Agents: ['PM', 'FrontendDev', 'BackendDev', 'TestEngineer']
   Max steps: 24

📝 User Task: "Build a Todo List web application with full CRUD functionality."
  ⏱️  Step 1: 📋 PM scheduled (inbox: 1)
  💬  Step 1: 📋 PM → #shared | ## 📋 PRD: Todo List Web Application  ### Overview Build a full-stack Todo List w...
  ⏱️  Step 2: 🎨 FrontendDev scheduled (inbox: 2)
  💬  Step 2: 🎨 FrontendDev → #shared | ## 🎨 Frontend Implementation Complete  ### Component Structure ``` src/ ├── App....
  ⏱️  Step 3: 📋 PM scheduled (inbox: 1)
  💬  Step 3: 📋 PM → #shared | Both frontend and backend implementations look great! 🎉  @TestEngineer Please re...
  ⏱️  Step 4: ⚙️ BackendDev scheduled (inbox: 4)
  💬  Step 4: ⚙️ BackendDev → #shared | ## ⚙️ Backend Implementation Complete  ### Project Structure ``` backend/ ├── ma...
  ⏱️  Step 5: 📋 PM scheduled (inbox: 1)
  🏁  Step 5: 📋 PM FINALIZED the project!

✅ Team finished!
   Stop reason: owner_finalized
   To

---
## Cell 6 — 📊 Full Visualization

In [6]:
render_all(result)

---
## Cell 7 — 🔍 Inspect Raw Data

Explore the raw events and transcript for debugging.

In [ ]:
# ── Events Table ───────────────────────────────────────────────────

print("📋 Event Log:")
print(f"{'Step':>5} | {'Type':<22} | {'Agent':<14} | Details")
print("-" * 80)

for ev in result["events"]:
    step = str(ev.get("step", "-")).rjust(5)
    etype = ev["type"].ljust(22)
    agent = ev.get("agent", "-").ljust(14)
    
    details = ""
    if ev["type"] == "message_published":
        details = f'#{ev.get("channel", "")} | {ev.get("content", "")[:50].replace(chr(10), " ")}...'
    elif ev["type"] == "scheduled":
        details = f'inbox_size={ev.get("inbox_size", 0)}'
    elif ev["type"] == "handoff":
        details = f'→ {ev.get("target", "?")}'
    elif ev["type"] == "finalized":
        details = f'{ev.get("content", "")[:50].replace(chr(10), " ")}...'
    
    print(f"{step} | {etype} | {agent} | {details}")

In [ ]:
# ── Transcript Messages ────────────────────────────────────────────

print("📨 Transcript (message envelopes):")
print()

for i, env in enumerate(result["transcript"]):
    cfg = _get_agent_cfg(env.get("sender", ""))
    print(f"── Message {i+1} ──────────────────────────────")
    print(f"  Sender:   {cfg['emoji']} {env.get('sender', '?')}")
    print(f"  Channel:  #{env.get('channel', '?')}")
    print(f"  Kind:     {env.get('kind', '?')}")
    print(f"  Step:     {env.get('step', '?')}")
    print(f"  Mentions: {env.get('mentions', [])}")
    print(f"  Content:  {env.get('content', '')[:100].replace(chr(10), ' ')}...")
    print()

---
## Cell 8 — 🔥 LLM Mode (Optional)

To use real LLM calls instead of simulated responses:
1. Set your API key below
2. Re-create agents **without** the monkeypatched `step()`
3. Run the team

⚠️ This will make actual API calls and consume tokens.

In [ ]:
# ── Uncomment and configure to use real LLM ───────────────────────

USE_LLM = False  # ← Set to True to enable

if USE_LLM:
    import os
    
    # Configure your provider
    PROVIDER = "openai"       # "openai", "anthropic", or "ollama"
    MODEL = "gpt-4o-mini"     # or "claude-sonnet-4-20250514", "llama3", etc.
    API_KEY = os.environ.get("OPENAI_API_KEY", "sk-your-key-here")
    
    # Re-create agents with real LLM config (no monkeypatch)
    llm_pm = Agent(
        name="PM",
        provider=PROVIDER, model=MODEL, api_key=API_KEY,
        instructions="""
You are the Product Manager. Your responsibilities:
1. Receive user requirements and decompose them into a clear PRD.
2. Assign frontend tasks to @FrontendDev and backend tasks to @BackendDev.
3. After both developers finish, ask @TestEngineer to validate.
4. After testing is complete, produce the final summary.
Be concise, structured, and professional.
""",
    )
    
    llm_frontend = Agent(
        name="FrontendDev",
        provider=PROVIDER, model=MODEL, api_key=API_KEY,
        instructions="""
You are the Frontend Developer. Read the PRD and implement React + TypeScript UI components.
Publish your completed code back to the shared channel. Be concise.
""",
    )
    
    llm_backend = Agent(
        name="BackendDev",
        provider=PROVIDER, model=MODEL, api_key=API_KEY,
        instructions="""
You are the Backend Developer. Read the PRD and design Python FastAPI endpoints + data models.
Publish your completed API design back to the shared channel. Be concise.
""",
    )
    
    llm_tester = Agent(
        name="TestEngineer",
        provider=PROVIDER, model=MODEL, api_key=API_KEY,
        instructions="""
You are the Test Engineer. Review frontend and backend deliverables.
Create a test plan, report results. Be thorough but concise.
""",
    )
    
    llm_team = Team(
        agents=[llm_pm, llm_frontend, llm_backend, llm_tester],
        owner="PM",
        channels={"shared": ["PM", "FrontendDev", "BackendDev", "TestEngineer"]},
        max_steps=24,
    )
    
    print("🔥 LLM Team created! Running with real API calls...")
    llm_result = llm_team.run(
        "Build a Todo List web application with full CRUD functionality.",
        callback=live_callback,
    )
    
    print(f"\n✅ LLM Team finished! Stop reason: {llm_result['stop_reason']}")
    render_all(llm_result)
else:
    print("ℹ️  LLM mode disabled. Set USE_LLM = True above to enable.")
    print("   Supported providers: openai, anthropic, ollama")

---

## 🎓 How It Works

### miso Team Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                        Team.run()                           │
│                                                             │
│  1. User message → entry_channel ("shared")                │
│  2. _deliver_channel_message() → all subscribers' inboxes  │
│  3. Loop:                                                   │
│     a. _select_next_agent() based on priority:             │
│        handoff > mention > owner_bonus > order             │
│     b. agent.step(inbox=...) → {publish, handoff, final}   │
│     c. Deliver published messages to channel subscribers    │
│     d. If final + owner → stop                             │
│  4. Return {events, transcript, final, stop_reason}        │
└─────────────────────────────────────────────────────────────┘
```

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Channel** | Named message bus. Subscribers receive copies of published messages. |
| **Envelope** | Message wrapper: `{sender, channel, content, kind, mentions, step}` |
| **Scheduling** | Priority-based: handoff targets > mentioned agents > owner > registration order |
| **Completion** | `owner_finalize` policy — only the owner agent can end the run |
| **Events** | Full audit trail: `scheduled`, `message_published`, `handoff`, `finalized`, `idle` |